# SFT: Supervised Fine-Tuning of Llama-3.2-1B on Automotive E/E Instruction Pairs
- Loads the CPT-adapted Llama-3.2-1B from Part 2
- Applies a narrow LoRA adapter
- Trains on 30 E/E instruction pairs using the Alpaca prompt template.
- Runs on Kaggle T4: Toy setup

## S0: Configuration

In [ ]:
GITHUB_REPO      = "https://github.com/tillacs/Domain-Adapted-LLM-Training-for-Automotive-E-E.git"
CPT_MODEL_PATH = "/kaggle/input/notebooks/tillkokemoor/cpt-notebook/cpt_merged"

PAIRS            = "repo/part3_sft/instruction_pairs.jsonl"
OUTPUT           = "/kaggle/working/sft_output"
MAX_SEQ_LENGTH   = 1024

LORA_RANK        = 16
LORA_ALPHA       = 32
LR               = 1e-4
TARGET_MODULES   = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"]
NUM_EPOCHS       = 3
SEED             = 42

## S1: Install unsloth

In [ ]:
!pip install unsloth unsloth_zoo bitsandbytes

## S2: Load Pretrained Model


In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = CPT_MODEL_PATH,   # merged 16-bit CPT model
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = False,
)
print("Loaded CPT model from:", CPT_MODEL_PATH)

## S3: Load and Format Instruction Pairs

Alpaca template: `### Instruction:\n{instruction}\n\n### Response:\n{response}`

This format is standard for Llama-family SFT and is supported natively by Unsloth's SFTTrainer.

In [ ]:
import json
from datasets import Dataset

!git clone -q {GITHUB_REPO} repo

def format_alpaca(row):
    return {"text": f"### Instruction:\n{row['instruction']}\n\n### Response:\n{row['response']}{tokenizer.eos_token}"}

with open(PAIRS) as f:
    pairs = [json.loads(line) for line in f if line.strip()]

train_ds = Dataset.from_list([format_alpaca(p) for p in pairs])

print(f"Train: {len(train_ds)}")
print(train_ds[0]["text"][:300])

## S4: LoRA Setup


In [ ]:
# No embed_tokens / lm_head as in CPT

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    lora_alpha     = LORA_ALPHA,
    target_modules = TARGET_MODULES,
    random_state   = SEED,
    
    use_rslora     = False,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    
)
model.print_trainable_parameters()

## S5: Training

In [ ]:
from unsloth import UnslothTrainer, UnslothTrainingArguments, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = UnslothTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_ds,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    args = UnslothTrainingArguments(
        output_dir                  = OUTPUT,
        num_train_epochs            = NUM_EPOCHS,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 1,
        learning_rate               = LR,
        lr_scheduler_type           = "cosine",
        warmup_steps                = 2,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        optim                       = "adamw_8bit",
        seed                        = SEED,
    )
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "### Instruction:\n",
    response_part    = "### Response:\n",
)

trainer.train()

## S6: Loss Curve

In [ ]:
import matplotlib.pyplot as plt
losses = [x["loss"] for x in trainer.state.log_history if "loss" in x]
plt.figure(figsize=(9, 3))
plt.plot(losses, lw=1.2); plt.xlabel("Step"); plt.ylabel("Loss")
plt.title("SFT Loss — Llama-3.2-1B, r=16"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## S7: Post SFT Inference

In [ ]:
FastLanguageModel.for_inference(model)

prompt = "### Instruction:\nWhich parts does the AUTOSAR software architecture consist of?\n\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 300,
    temperature    = 0,      # greedy — deterministic output for comparison
    do_sample      = False,
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## S8: Save Model

In [ ]:
# merged 16-bit model to Kaggle-Output
model.save_pretrained_merged(OUTPUT, tokenizer, save_method="merged_16bit")
print("Saved merged CPT+SFT model to:", OUTPUT)